# Implement a Transformer Decoder Block from Scratch
### Problem Statement
The Transformer Decoder Block is the core repeating unit in decoder-only LLMs (GPT, Llama, Mistral, etc.). Each block combines all the components you've built in this series:

```
x → RMSNorm → Multi-Head Attention → + (residual) → RMSNorm → SwiGLU FFN → + (residual) → output
```

This is the **Pre-Norm** architecture used by modern LLMs. A full model stacks N of these blocks (e.g., 32 for Llama 2 7B).

Your goal is to implement a complete decoder block using the components from previous exercises, with causal masking for autoregressive generation.

---

### Requirements

1. **Components**
   - RMSNorm (from `08-RMS-Norm`)
   - Multi-Head Attention (from `04-Multi-Head-Attention`) with **causal mask**
   - SwiGLU Feed-Forward Network (from `10-Feed-Forward-Network`)
   - Residual connections (from `09-Residual-Connection`)

2. **Causal Masking**
   - Generate a lower-triangular mask so each token can only attend to itself and previous tokens.
   - This enforces the autoregressive property.

3. **Architecture (Pre-Norm)**
   ```
   h = x + Attention(RMSNorm(x), causal_mask)
   output = h + FFN(RMSNorm(h))
   ```

4. **Validate**
   - Output shape must be `(batch_size, seq_len, d_model)`.
   - Causal mask must prevent future token leakage.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Pre-Norm architecture (norm before sublayer, not after).
- ✅ Causal masking must be correct.
- ✅ All sub-components should be implemented from scratch (no `nn.TransformerDecoderLayer`).

---

<details>
  <summary>💡 Hint</summary>

  - Causal mask: `torch.tril(torch.ones(seq_len, seq_len))` — lower triangular matrix of ones.
  - In the attention scores, mask positions with 0 should be filled with `-inf` before softmax.
  - The block has no learnable parameters of its own — it just orchestrates sub-modules.
  - Remember: Pre-Norm means `x + sublayer(norm(x))`, not `norm(x + sublayer(x))`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Building blocks (from previous exercises)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention with causal mask support."""
    def __init__(self, num_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads

        self.Q_w = nn.Linear(d_model, d_model, bias=False)
        self.K_w = nn.Linear(d_model, d_model, bias=False)
        self.V_w = nn.Linear(d_model, d_model, bias=False)
        self.W_out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        """Self-attention: q=k=v=x"""
        batch_size, seq_len, _ = x.shape

        Q = self.Q_w(x)
        K = self.K_w(x)
        V = self.V_w(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)

        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_out(output)

In [ ]:
class SwiGLUFeedForward(nn.Module):
    """SwiGLU FFN."""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or int(2 / 3 * 4 * d_model)
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

In [ ]:
def create_causal_mask(seq_len):
    """Returns a lower-triangular boolean mask of shape (1, 1, seq_len, seq_len)."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len) for broadcasting

In [ ]:
class TransformerDecoderBlock(nn.Module):
    """
    A single Transformer Decoder Block (Pre-Norm architecture).
    x → RMSNorm → Attention → + residual → RMSNorm → FFN → + residual → output
    """
    def __init__(self, num_heads: int, d_model: int, d_ff: int = None):
        super().__init__()
        self.attn = MultiHeadAttention(num_heads, d_model)
        self.ffn = SwiGLUFeedForward(d_model, d_ff)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)

    def forward(self, x, mask=None):
        """
        Args:
            x (Tensor): Input of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Causal mask

        Returns:
            Tensor: Output of shape (batch_size, seq_len, d_model)
        """
        # Pre-Norm: norm before sublayer, residual on clean path
        h = x + self.attn(self.norm1(x), mask)   # attention + residual
        output = h + self.ffn(self.norm2(h))      # FFN + residual
        return output

## 📚 The Complete Decoder Block

### Architecture Diagram

```
        ┌───────────────────────────────────┐
        │                                   │
  x ────┤─→ RMSNorm ─→ Multi-Head Attn ─→ + ─→ h
        │        (causal mask)              │
        └───────────────────────────────────┘
        ┌───────────────────────────────────┐
        │                                   │
  h ────┤─→ RMSNorm ─→ SwiGLU FFN ──────→ + ─→ output
        │                                   │
        └───────────────────────────────────┘
```

### How It Maps to Real Models

| Component | Original Transformer | Llama 2/3 | GPT-2 |
|-----------|---------------------|-----------|-------|
| Norm | LayerNorm (Post) | RMSNorm (Pre) | LayerNorm (Pre) |
| Attention | MHA | GQA | MHA |
| FFN | ReLU | SwiGLU | GELU |
| Position | Sinusoidal | RoPE | Learned |
| Bias | Yes | No | Yes |

### Causal Mask Visualization

For seq_len=4, the causal mask looks like:
```
  t1 t2 t3 t4
t1 [1  0  0  0]   ← t1 can only see t1
t2 [1  1  0  0]   ← t2 can see t1, t2
t3 [1  1  1  0]   ← t3 can see t1, t2, t3
t4 [1  1  1  1]   ← t4 can see all tokens
```

Positions with 0 are filled with `-inf` before softmax, making their attention weight exactly 0.

### Parameter Breakdown

For a block with `d_model=4096, num_heads=32, d_ff=11008` (Llama 2 7B):

| Component | Parameters |
|-----------|----------|
| Attention (Q,K,V,O) | 4 × 4096² = 67M |
| FFN (gate, up, down) | 3 × 4096 × 11008 = 135M |
| RMSNorm × 2 | 2 × 4096 = 8K |
| **Block total** | **~202M** |
| **32 blocks** | **~6.5B** (+ embeddings = ~7B) |

In [ ]:
# Test
torch.manual_seed(42)
batch_size, seq_len, d_model, num_heads = 2, 6, 32, 4

block = TransformerDecoderBlock(num_heads, d_model)
x = torch.randn(batch_size, seq_len, d_model)
mask = create_causal_mask(seq_len)

output = block(x, mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Mask shape:   {mask.shape}")
assert output.shape == x.shape

# Count parameters
total_params = sum(p.numel() for p in block.parameters())
print(f"\nTotal parameters: {total_params:,}")
print("\n✅ Transformer Decoder Block works correctly!")

In [ ]:
# Verify causal masking: changing a future token should NOT affect earlier outputs
torch.manual_seed(42)
x1 = torch.randn(1, 4, d_model)
x2 = x1.clone()
x2[:, 3, :] = torch.randn(d_model)  # Change the last token

mask = create_causal_mask(4)
block.eval()
with torch.no_grad():
    out1 = block(x1, mask)
    out2 = block(x2, mask)

# First 3 tokens should be identical (they can't see token 4)
print("Outputs match for tokens that can't see the changed future token:")
for t in range(4):
    match = torch.allclose(out1[:, t, :], out2[:, t, :], atol=1e-6)
    changed = "(changed)" if t == 3 else ""
    print(f"  Token {t}: {'✅ match' if match else '❌ different'} {changed}")

print("\n✅ Causal masking verified — future tokens don't leak!")

In [ ]:
# Bonus: Stack multiple blocks to form a mini decoder
num_layers = 4
blocks = nn.ModuleList([TransformerDecoderBlock(num_heads, d_model) for _ in range(num_layers)])

h = x
for blk in blocks:
    h = blk(h, mask)

print(f"Input:  {x.shape}")
print(f"After {num_layers} blocks: {h.shape}")
total = sum(p.numel() for p in blocks.parameters())
print(f"Total params ({num_layers} layers): {total:,}")